<a href="https://colab.research.google.com/github/isti2415/CSC551-Data-Mining/blob/main/Assignment%201/Assignment1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1: Python / NumPy basics

**Objective:** dataset characteristics, data visualization, data analysis.

Dataset: **Date Fruit** (Koklu et al.). Every row is one date fruit image; the columns are size, shape and colour
features measured from the image, and the last column `Class` is the type of date.

## Setup

- `pandas` to read the Excel file into a data frame
- `numpy` for arrays and vectorised (linear algebra) maths
- `plotly` for interactive plots (rotate, zoom, hover)
- `kagglehub` to download the dataset, `openpyxl` so pandas can read `.xlsx`

In [ ]:
%pip install -q kagglehub pandas numpy plotly openpyxl nbformat

In [2]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import kagglehub

/home/user/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Task 1: Download the dataset

> Download the date fruit dataset mentioned in the group information sheet. Carefully read the description of the
> dataset. Download the csv or the xlsx file.

The dataset has 898 date fruit images of 7 types (BERHI, DEGLET, DOKOL, IRAQI, ROTANA, SAFAVI, SOGAY). For each image,
34 features were extracted: 12 size/shape features (area, perimeter, axes, ...), 4 shape factors and 18 colour features.
Size features are measured in **pixels** (areas in px²); ratios such as eccentricity and solidity have no unit.

In [3]:
path = kagglehub.dataset_download("muratkokludataset/date-fruit-datasets")
print("Path to dataset files:", path)

for folder, _, files in os.walk(path):
    for f in files:
        print(os.path.join(folder, f))

Path to dataset files: /root/.cache/kagglehub/datasets/muratkokludataset/date-fruit-datasets/versions/1
/root/.cache/kagglehub/datasets/muratkokludataset/date-fruit-datasets/versions/1/Date_Fruit_Datasets/Date_Fruit_Datasets_Citation_Request.txt
/root/.cache/kagglehub/datasets/muratkokludataset/date-fruit-datasets/versions/1/Date_Fruit_Datasets/Date_Fruit_Datasets.xlsx
/root/.cache/kagglehub/datasets/muratkokludataset/date-fruit-datasets/versions/1/Date_Fruit_Datasets/Date_Fruit_Datasets.arff


## Task 2: Read the xlsx file

> Read the xlsx file using panda data frame.

`shape` shows (rows, columns): 898 data points, 34 features + 1 class column.

In [4]:
xlsx_file = [os.path.join(folder, f)
             for folder, _, files in os.walk(path)
             for f in files if f.endswith(".xlsx")][0]

df_full = pd.read_excel(xlsx_file)
print(df_full.shape)
df_full.head()

(898, 35)


,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [5]:
print(df_full["Class"].value_counts())

Class
DOKOL     204
SAFAVI    199
ROTANA    166
DEGLET     98
SOGAY      94
IRAQI      72
BERHI      65
Name: count, dtype: int64


## Task 3: Keep the first three dimensions and the class label

> Now keep first, second, and thirds dimensions (real values) only. You should also keep the last column because it
> contains class label. So you will have four columns in your data frame.

The dataset has no ID / serial-number column, so the first three columns `AREA`, `PERIMETER`, `MAJOR_AXIS` are already
real-valued features. We also keep `df_full`, because tasks 15 and 16 need the first 7 dimensions.

In [6]:
df = df_full.iloc[:, [0, 1, 2, -1]]
feature_names = list(df.columns[:3])
print(df.dtypes)
df.head()

AREA            int64
PERIMETER     float64
MAJOR_AXIS    float64
Class             str
dtype: object


,AREA,PERIMETER,MAJOR_AXIS,Class
0,422163,2378.908,837.8484,BERHI
1,338136,2085.144,723.8198,BERHI
2,526843,2647.394,940.7379,BERHI
3,416063,2351.210,827.9804,BERHI
4,347562,2160.354,763.9877,BERHI


## Task 4: Convert to a NumPy array

> Convert your data frame into numpy array. If your class labels are strings, use numeric class label such as 0, 1, 2
> for the classes. Store the string class name in a list so that you can use them using their class index.

- `data` is the 4-column data frame as a NumPy array.
- `X` is the $n \times 3$ data matrix $\mathbf{D}$ (the three numeric features).
- `y` holds the numeric class label 0, 1, 2, ... of every point.
- `class_names[k]` gives the string name of class `k`.

In [7]:
data = df.to_numpy()

X = data[:, :3].astype(float)
class_names, y = np.unique(data[:, 3], return_inverse=True)
class_names = list(class_names)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", class_names)
print("Class of first row:", y[0], "->", class_names[y[0]])

X shape: (898, 3)
y shape: (898,)
Classes: ['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY']
Class of first row: 0 -> BERHI


## Task 5: Plotly for interactive plots

> You should use plotly library to create interactive plots so that we can rotate the plot.
> Reference: https://plotly.com/python/3d-scatter-plots/

Every class gets its own **shape**, chosen only from the class label `k`. Plotly 3D scatter plots do not support
triangles (only circle, square, diamond, cross, x and their open versions), so the 3D list is different. Each class also
keeps the same colour in every plot.

In [8]:
symbols_2d = ["triangle-up", "square", "circle", "diamond", "star", "cross", "x"]
symbols_3d = ["circle", "square", "diamond", "cross", "x", "circle-open", "square-open"]
colors = px.colors.qualitative.Plotly[:len(class_names)]

for k, name in enumerate(class_names):
    print(k, name, "2D:", symbols_2d[k], "3D:", symbols_3d[k])

0 BERHI 2D: triangle-up 3D: circle
1 DEGLET 2D: square 3D: square
2 DOKOL 2D: circle 3D: diamond
3 IRAQI 2D: diamond 3D: cross
4 ROTANA 2D: star 3D: x
5 SAFAVI 2D: cross 3D: circle-open
6 SOGAY 2D: x 3D: square-open


## Task 6: 2D plot of the first two dimensions

> Plot the data points using the first two dimensions in 2d plots. Use different shapes (triangle, square, circle,
> diamond, ...) to plot data points for different classes. You should use the class information from class label and
> use them when you decide on shapes. 2D plot must be interactive.

The figure is built in a function so that task 8 can reuse exactly the same points.

In [9]:
def make_2d_figure():
    fig = go.Figure()
    for k, name in enumerate(class_names):
        pts = X[y == k]
        fig.add_trace(go.Scatter(
            x=pts[:, 0], y=pts[:, 1], mode="markers", name=name,
            marker=dict(symbol=symbols_2d[k], size=8, color=colors[k],
                        line=dict(color="white", width=0.5))))
    fig.update_layout(title="Date fruit: " + feature_names[0] + " vs " + feature_names[1],
                      xaxis_title=feature_names[0] + " (px²)",
                      yaxis_title=feature_names[1] + " (px)",
                      width=900, height=600)
    return fig

make_2d_figure().show()

## Task 7: 3D plot of the first three dimensions

> Plot the data points using the first three dimensions in 3d plots. Use different shapes to plot data points for
> different classes. 3D plot must be interactive (use plotly).

Drag to rotate, scroll to zoom.

In [10]:
def make_3d_figure():
    fig = go.Figure()
    for k, name in enumerate(class_names):
        pts = X[y == k]
        fig.add_trace(go.Scatter3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="markers", name=name,
            marker=dict(symbol=symbols_3d[k], size=4, color=colors[k])))
    fig.update_layout(title="Date fruit: first three dimensions",
                      scene=dict(xaxis_title=feature_names[0] + " (px²)",
                                 yaxis_title=feature_names[1] + " (px)",
                                 zaxis_title=feature_names[2] + " (px)"),
                      width=900, height=700)
    return fig

make_3d_figure().show()

## Task 8: 2D plot with the class means

> Again do task 6 (2D). And calculate the mean data point for each class and show them with similar shape with the
> larger size. Plot the means on the plot you got at task 6. You MUST keep the original points as they are.

The mean point of class $k$ with $n_k$ points is $\boldsymbol{\mu}_k = \frac{1}{n_k}\sum_{\mathbf{x}_i \in k} \mathbf{x}_i$.
We start from the task 6 figure (points unchanged) and add one big marker per class, same shape and colour, with a black
outline so it stands out.

In [11]:
means = np.array([X[y == k].mean(axis=0) for k in range(len(class_names))])
print(pd.DataFrame(means, index=class_names, columns=feature_names))

fig = make_2d_figure()
for k, name in enumerate(class_names):
    fig.add_trace(go.Scatter(
        x=[means[k, 0]], y=[means[k, 1]], mode="markers", name=name + " mean",
        marker=dict(symbol=symbols_2d[k], size=22, color=colors[k],
                    line=dict(color="black", width=2))))
fig.update_layout(title="Task 6 plot with the class means (large markers)")
fig.show()

                 AREA    PERIMETER  MAJOR_AXIS
BERHI   405213.446154  2321.504669  815.370329
DEGLET  233743.357143  1831.441983  680.222427
DOKOL   145147.421569  1436.321190  537.843773
IRAQI   433983.583333  2411.046333  836.104518
ROTANA  387115.867470  2376.310050  832.475543
SAFAVI  334367.783920  2329.186030  881.385573
SOGAY   286872.957447  2051.277354  755.978483


## Task 9: 3D plot with the class means

> Repeat the task 8 for three dimensional plot.

In [12]:
fig = make_3d_figure()
for k, name in enumerate(class_names):
    fig.add_trace(go.Scatter3d(
        x=[means[k, 0]], y=[means[k, 1]], z=[means[k, 2]], mode="markers",
        name=name + " mean",
        marker=dict(symbol=symbols_3d[k], size=12, color=colors[k],
                    line=dict(color="black", width=3))))
fig.update_layout(title="Task 7 plot with the class means (large markers)")
fig.show()

## Task 10: Histograms

> Plot histogram for each of the first three dimensions. One plot for each dimension.

In [13]:
units = ["px²", "px", "px"]
for i in range(3):
    fig = px.histogram(x=X[:, i], nbins=40, title="Histogram of " + feature_names[i],
                       labels={"x": feature_names[i] + " (" + units[i] + ")"})
    fig.update_layout(yaxis_title="Count", bargap=0.05, width=800, height=450)
    fig.show()

## Task 11: Standard deviation (vectorised, no loop)

> Calculate standard deviation (sigma) for each of the three dimensions. You must use vectorized code (linear
> algebra – no loop) to calculate standard deviation.

From the lectures (Lecture 2, slide 7 and Lecture 2.2, slides 12–13):

- mean vector: $\hat{\boldsymbol{\mu}} = \frac{1}{n}\sum_i \mathbf{x}_i = \frac{1}{n}\mathbf{D}^T\mathbf{1}$
- centred data matrix: $\mathbf{Z} = \mathbf{D} - \mathbf{1}\cdot\hat{\boldsymbol{\mu}}^T$
- sample variance of attribute $j$ is the squared length of its centred column $Z_j$, divided by $n$:
  $\hat{\sigma}_j^2 = \frac{1}{n}\|Z_j\|^2 = \frac{1}{n} Z_j^T Z_j$

$Z_j^T Z_j$ for all $j$ at once is the **diagonal** of $\mathbf{Z}^T\mathbf{Z}$, so
$\hat{\boldsymbol{\sigma}} = \sqrt{\operatorname{diag}(\mathbf{Z}^T\mathbf{Z}) / n}$, with no loop.
`np.std` (which also divides by $n$) is used only to check the answer.

In [14]:
n = X.shape[0]
ones = np.ones(n)

mu = X.T @ ones / n                 # mean vector (1/n) D^T 1
Z = X - np.outer(ones, mu)          # centred data  Z = D - 1 mu^T
sigma = np.sqrt(np.diag(Z.T @ Z) / n)

print(pd.DataFrame({"mean": mu,
                    "std (vectorised)": sigma,
                    "std (np.std check)": np.std(X, axis=0)}, index=feature_names))

                     mean  std (vectorised)  std (np.std check)
AREA        298295.207127     107185.475338       107185.475338
PERIMETER     2057.660953        409.784104          409.784104
MAJOR_AXIS     750.811994        143.979092          143.979092


## Task 12: Box plots of the first three dimensions

> Draw box plots for all data points for first three dimensions. X axis have dimensions. The Y axis will have the box
> plots. (See slide 7, lecture 1)

A box plot shows the five-number summary: minimum, Q1, median, Q3, maximum. Points more than 1.5 × IQR beyond the box
are drawn as outliers.

AREA (~300 000 px²) is far larger than PERIMETER and MAJOR_AXIS (~1 000 px), so on raw values the other two boxes become
flat lines. Lecture 2 (slide 11) says to **normalise** attributes in such cases, so the second plot uses range
normalisation $x' = \frac{x - \min(x)}{\max(x) - \min(x)}$, which puts every dimension on $[0, 1]$.

In [15]:
fig = go.Figure()
for i in range(3):
    fig.add_trace(go.Box(y=X[:, i], name=feature_names[i], boxpoints="outliers"))
fig.update_layout(title="Box plots of the first three dimensions (raw values)",
                  xaxis_title="Dimension", yaxis_title="Value", showlegend=False,
                  width=800, height=500)
fig.show()

In [16]:
X_min, X_max = X.min(axis=0), X.max(axis=0)
X_norm = (X - X_min) / (X_max - X_min)

fig = go.Figure()
for i in range(3):
    fig.add_trace(go.Box(y=X_norm[:, i], name=feature_names[i], boxpoints="outliers"))
fig.update_layout(title="Box plots of the first three dimensions (range-normalised to [0, 1])",
                  xaxis_title="Dimension", yaxis_title="Normalised value", showlegend=False,
                  width=800, height=500)
fig.show()

## Task 13: Box plots for each class separately

> Now, draw box plots for each class separately. In each figure, you have boxes for each class for three dimensions.

One figure per class, with the three dimensions on the X axis, like task 12. Values are range-normalised with the
min and max of the **whole** dataset, and every figure uses the same Y range [0, 1], so a box can be compared directly
between classes (e.g. DOKOL has much smaller fruit than IRAQI).

In [17]:
for k, name in enumerate(class_names):
    fig = go.Figure()
    for i in range(3):
        fig.add_trace(go.Box(y=X_norm[y == k, i], name=feature_names[i],
                             marker_color=colors[k], boxpoints="outliers"))
    fig.update_layout(title="Class " + name + " (" + str((y == k).sum()) + " points)",
                      xaxis_title="Dimension", yaxis_title="Normalised value",
                      yaxis_range=[-0.05, 1.05], showlegend=False, width=700, height=420)
    fig.show()

## Task 14: Scatter matrix (3 × 3)

> Make matrix plots for the entire dataset for 3 dimensions (see slide 21, lecture 1). So the whole figure will have
> 3x3 scatter matrices.

Cell (row $i$, column $j$) plots dimension $j$ against dimension $i$, so all pairs appear in one figure. Points are
coloured and shaped by class.

In [18]:
fig = px.scatter_matrix(df, dimensions=feature_names, color="Class", symbol="Class",
                        color_discrete_sequence=colors, symbol_sequence=symbols_2d,
                        category_orders={"Class": class_names},
                        title="Scatter matrix of the first three dimensions")
fig.update_traces(marker=dict(size=4))
fig.update_layout(width=900, height=850)
fig.show()

## Task 15: Circle segments (first 7 dimensions)

> Draw plots to represent data points in circle segment (see lecture slide 19a) for first 7 real valued dimensions.

Circle segments is a **pixel-oriented** technique:

1. The circle is divided into 7 equal segments, **one segment per dimension**.
2. Every data point is drawn as **one pixel in every segment**, at the **same position** in each segment. The pixel's
   **colour** is the value of that dimension.
3. Points are placed from the centre outwards: ring 1 holds 1 pixel, ring 2 holds 2 pixels, and so on, so the pixels
   fill each wedge.

The points are sorted by class (then by AREA inside a class), so each class occupies a band of rings at the same radius
in all 7 segments. Values are range-normalised to [0, 1] (Lecture 2, slide 11) so one colour scale fits all 7
dimensions. Hover over a pixel to see its class and real value.

In [19]:
first7 = df_full.iloc[:, :7]
names7 = list(first7.columns)
units7 = ["px²", "px", "px", "px", "ratio", "px", "ratio"]
D7 = first7.to_numpy(dtype=float)
N7 = (D7 - D7.min(axis=0)) / (D7.max(axis=0) - D7.min(axis=0))
print(names7)

['AREA', 'PERIMETER', 'MAJOR_AXIS', 'MINOR_AXIS', 'ECCENTRICITY', 'EQDIASQ', 'SOLIDITY']


In [20]:
def circle_segments(rows, fig, polar="polar", show_labels=True):
    """Draw the points `rows` (indices into D7, in drawing order) as circle segments on subplot `polar`."""
    m = len(rows)
    K = int(np.ceil((np.sqrt(8 * m + 1) - 1) / 2))        # rings so that 1 + 2 + ... + K >= m
    ring = np.repeat(np.arange(K), np.arange(1, K + 1))[:m]
    slot = np.concatenate([np.arange(r + 1) for r in range(K)])[:m]
    seg = 360 / 7

    for d in range(7):
        fig.add_trace(go.Barpolar(
            theta=d * seg + (slot + 0.5) * seg / (ring + 1),
            width=seg / (ring + 1),
            base=ring / K, r=np.full(m, 1 / K),
            marker=dict(color=N7[rows, d], coloraxis="coloraxis", line_width=0),
            customdata=np.column_stack([np.array(class_names)[y[rows]], D7[rows, d]]),
            hovertemplate=names7[d] + ": %{customdata[1]:.4g}<br>class %{customdata[0]}<extra></extra>",
            showlegend=False, subplot=polar))

    # thin white spokes between the segments
    for d in range(7):
        fig.add_trace(go.Scatterpolar(r=[0, 1], theta=[d * seg] * 2, mode="lines",
                                      line=dict(color="white", width=2), hoverinfo="skip",
                                      showlegend=False, subplot=polar))
    fig.layout[polar].update(
        bargap=0, hole=0, bgcolor="white",
        radialaxis=dict(visible=False, range=[0, 1]),
        angularaxis=dict(tickvals=[(d + 0.5) * seg for d in range(7)],
                         ticktext=names7 if show_labels else [""] * 7,
                         showgrid=False, direction="clockwise", rotation=90))


order = np.lexsort((D7[:, 0], y))          # sort by class, then by AREA
fig = go.Figure()
circle_segments(order, fig)
fig.update_layout(title="Circle segments: all 898 data points, 7 dimensions (sorted by class)",
                  coloraxis=dict(colorscale="Viridis", cmin=0, cmax=1,
                                 colorbar=dict(title="normalised<br>value", x=1.12)),
                  width=900, height=800)
fig.show()

The same technique with **one circle per class** makes the class differences easier to see: e.g. DOKOL is dark
(small values) in all size segments, while IRAQI and BERHI are bright (large fruit). ECCENTRICITY and SOLIDITY behave
differently from the size features, which shows they carry different information.

In [21]:
fig = make_subplots(rows=2, cols=4, specs=[[{"type": "polar"}] * 4, [{"type": "polar"}] * 3 + [None]],
                    subplot_titles=class_names, horizontal_spacing=0.06, vertical_spacing=0.12)
for k in range(len(class_names)):
    rows = np.where(y == k)[0]
    rows = rows[np.argsort(D7[rows, 0])]
    circle_segments(rows, fig, polar="polar" + (str(k + 1) if k else ""), show_labels=(k == 0))

fig.update_layout(title="Circle segments, one circle per class",
                  coloraxis=dict(colorscale="Viridis", cmin=0, cmax=1,
                                 colorbar=dict(title="normalised<br>value")),
                  width=1200, height=700)
fig.show()

## Task 16: Parallel coordinates (first 7 dimensions)

> Draw parallel coordinates for the entire dataset for first 7 dimensions (except class label). The line colors will
> depend on the class labels. You will have 7 equidistant axes that are parallel. Show the axes units and axes names.
> Visually observe the mean and std for each dimension and match with your previous calculation.

Each data point is one line crossing the 7 parallel axes. The line colour is the class. Each axis shows its name, its
unit and its real values, so the mean and spread can be read off it.

In [22]:
df7 = first7.copy()
df7["class_index"] = y
labels = {c: c + " (" + u + ")" for c, u in zip(names7, units7)}

# a stepped colour scale gives exactly one solid colour per class
step = []
for k, c in enumerate(colors):
    step += [(k / len(colors), c), ((k + 1) / len(colors), c)]

fig = px.parallel_coordinates(df7, dimensions=names7, color="class_index", labels=labels,
                              color_continuous_scale=step, range_color=[-0.5, len(colors) - 0.5],
                              title="Parallel coordinates of the first 7 dimensions")
fig.update_coloraxes(colorbar=dict(title="Class", tickvals=list(range(len(colors))), ticktext=class_names))
fig.update_layout(width=1250, height=550, margin=dict(l=80, r=80))
fig.show()

Mean and standard deviation of the 7 dimensions, computed with the same vectorised formula as task 11, to compare with
the plot:

In [23]:
n7 = D7.shape[0]
mu7 = D7.T @ np.ones(n7) / n7
Z7 = D7 - np.outer(np.ones(n7), mu7)
std7 = np.sqrt(np.diag(Z7.T @ Z7) / n7)

stats7 = pd.DataFrame({"unit": units7, "min": D7.min(axis=0), "mean": mu7, "std": std7,
                       "max": D7.max(axis=0)}, index=names7)
stats7

,unit,min,mean,std,max
AREA,px²,1987.0000,298295.207127,107185.475338,546063.0000
PERIMETER,px,911.8280,2057.660953,409.784104,2811.9971
MAJOR_AXIS,px,336.7227,750.811994,143.979092,1222.7230
MINOR_AXIS,px,2.2832,495.872785,114.205275,766.4536
ECCENTRICITY,ratio,0.3448,0.737468,0.088678,1.0000
EQDIASQ,px,50.2984,604.577938,119.527281,833.8279
SOLIDITY,ratio,0.8366,0.981840,0.018147,0.9974


### Reading the result

- On each axis most lines bunch up around the **mean** in the table, and the band where most lines cross is roughly
  mean ± std. For example, AREA lines gather around 300 000 px², spread about ±107 000, and PERIMETER around 2 058 ± 410 px.
- The means and standard deviations of the first three dimensions are the same as in task 11.
- MINOR_AXIS and ECCENTRICITY have a few extreme lines (min 2.3 px, eccentricity 1.0); these are the outliers that also
  show up in the box plots.
- Axes where colours separate cleanly (AREA, PERIMETER, EQDIASQ: DOKOL low, IRAQI/BERHI high) are the most useful
  for telling the classes apart.